# Loto 6/49 — Data Collection and Initial Cleaning

This notebook collects historical **Romanian Loto 6/49** draw results from the official lottery website.

## Scope

For each draw, the notebook extracts only:

- the draw date;
- the six winning Loto 6/49 numbers.

Other games, prize categories, jackpot values and winner counts are intentionally excluded.

## Data source

The official results page accepts an HTTP `POST` request containing:

- `select-year`;
- `select-month`.

The winning numbers are displayed as image files. Their numeric values are available directly in the image paths, for example:

```text
.../images/bile/24.png
```

Therefore, OCR is not required.

In [32]:
# Core libraries used for HTTP requests, HTML parsing and tabular data.
import re
from typing import List

import pandas as pd
import requests
from bs4 import BeautifulSoup

OFFICIAL_RESULTS_URL = (
    "https://www.loto.ro/loto-new/"
    "newLotoSiteNexioFinalVersion/web/app2.php/"
    "jocuri/649_si_noroc/rezultate_extragere.html"
)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
    )
}

## Monthly extraction function

The function below:

1. requests one selected month and year;
2. locates each Loto 6/49 result block;
3. extracts the date from the result header;
4. extracts the six numbers from the image filenames;
5. returns a chronologically sorted `pandas.DataFrame`.

Invalid or incomplete result blocks are skipped rather than added as corrupted rows.

In [33]:
def extract_loto_649_month(year: int, month: int) -> pd.DataFrame:
    """Extract all Romanian Loto 6/49 draws for one month.

    Parameters
    ----------
    year:
        Four-digit year available on the official results page.
    month:
        Calendar month from 1 to 12.

    Returns
    -------
    pandas.DataFrame
        One row per draw with the date and six winning numbers.
    """
    if not 1 <= month <= 12:
        raise ValueError("month must be between 1 and 12")

    response = requests.post(
        OFFICIAL_RESULTS_URL,
        data={
            "select-year": str(year),
            "select-month": str(month),
        },
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    records: List[dict] = []

    # Each result block contains one draw date and one group of six ball images.
    result_blocks = soup.select(
        "div.rezultate-extrageri-content.resultDiv"
    )

    for block in result_blocks:
        date_element = block.select_one(
            "div.button-open-details span"
        )
        number_images = block.select(
            "div.numere-extrase img"
        )

        if date_element is None or len(number_images) != 6:
            continue

        numbers = []

        for image in number_images:
            image_path = image.get("src", "")
            match = re.search(r"/bile/(\d+)\.png$", image_path)

            if match:
                numbers.append(int(match.group(1)))

        if len(numbers) != 6:
            continue

        records.append(
            {
                "draw_date": date_element.get_text(strip=True),
                "number_1": numbers[0],
                "number_2": numbers[1],
                "number_3": numbers[2],
                "number_4": numbers[3],
                "number_5": numbers[4],
                "number_6": numbers[5],
            }
        )

    columns = [
        "draw_date",
        "number_1",
        "number_2",
        "number_3",
        "number_4",
        "number_5",
        "number_6",
    ]
    df = pd.DataFrame(records, columns=columns)

    if df.empty:
        return df

    df["draw_date"] = pd.to_datetime(
        df["draw_date"],
        format="%d.%m.%Y",
        errors="raise",
    )

    return (
        df.drop_duplicates(subset="draw_date")
        .sort_values("draw_date")
        .reset_index(drop=True)
    )

## Extraction test

June 2026 is used as a reproducible test month. The expected output is one row per draw, ordered from the earliest to the latest date.

In [34]:
june_2026_df = extract_loto_649_month(2026, 6)

display(june_2026_df)
print(f"Number of draws: {len(june_2026_df)}")

,draw_date,number_1,number_2,number_3,number_4,number_5,number_6
0,2026-06-04,40,43,5,21,14,34
1,2026-06-07,48,18,22,27,46,15
2,2026-06-11,48,43,25,13,49,44
3,2026-06-14,3,25,7,10,5,18
4,2026-06-18,20,27,12,36,2,15
5,2026-06-21,42,6,21,35,14,37
6,2026-06-25,10,7,4,47,21,26
7,2026-06-28,47,8,4,44,31,23


Number of draws: 8


## Data-quality checks

Before using the data for statistical analysis or machine learning, each monthly dataset is checked for:

- exactly six values per draw;
- numbers restricted to the valid interval 1–49;
- six distinct numbers within every draw;
- unique draw dates;
- chronological ordering.

In [35]:
def validate_loto_649_data(df: pd.DataFrame) -> None:
    """Raise an assertion error when the extracted dataset is invalid."""
    number_columns = [f"number_{index}" for index in range(1, 7)]

    assert list(df.columns) == ["draw_date", *number_columns]
    assert df["draw_date"].notna().all()
    assert df["draw_date"].is_unique
    assert df["draw_date"].is_monotonic_increasing
    assert df[number_columns].notna().all().all()
    assert df[number_columns].apply(
        lambda row: row.between(1, 49).all(),
        axis=1,
    ).all()
    assert df[number_columns].apply(
        lambda row: row.nunique() == 6,
        axis=1,
    ).all()


validate_loto_649_data(june_2026_df)
print("Validation passed.")

Validation passed.


## Initial conclusion

The official website can be queried programmatically without browser automation.

The relevant information is available directly in the returned HTML:

- draw dates are stored as text;
- winning values are encoded in the ball-image filenames;
- one HTTP request retrieves all draws for a selected month.

This monthly extractor will be reused in the next step to collect the complete historical dataset across all available years and months.


## Full historical data collection

The monthly extraction function is now reused to collect all available Loto 6/49 draws across multiple years.

The collection process:

1. iterates through every year and month in the selected interval;
2. skips future months automatically;
3. calls the official results page once per month;
4. stores valid monthly datasets;
5. records the extraction status and number of draws for every processed month;
6. combines all monthly results into one chronological dataset;
7. removes duplicate draw dates.

A short delay is added between requests to avoid sending a large number of requests to the official website in a very short time.

Months may contain different numbers of draws. This is expected and does not require missing rows to be created. The scraper collects all draws actually published for each month.

A month returning zero draws is not automatically considered an error. For example, the current month may contain fewer draws, while future months are excluded entirely.

In [38]:
import time
from datetime import date
from typing import Tuple


def extract_loto_649_history(
    start_year: int = 1998,
    end_year: int | None = None,
    request_delay: float = 0.5
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract all available Romanian Loto 6/49 draws
    between the selected start and end years.

    Parameters
    ----------
    start_year:
        First year to process.

    end_year:
        Last year to process. If omitted, the current year is used.

    request_delay:
        Number of seconds to wait between monthly requests.

    Returns
    -------
    full_history_df:
        Combined dataset containing all extracted draws.

    extraction_log_df:
        Log containing the extraction status and draw count
        for every processed month.
    """
    today = date.today()

    if end_year is None:
        end_year = today.year

    if start_year > end_year:
        raise ValueError(
            "start_year must be smaller than or equal to end_year"
        )

    if request_delay < 0:
        raise ValueError(
            "request_delay cannot be negative"
        )

    monthly_datasets = []
    extraction_log = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):

            # Future months are not requested.
            if year > today.year or (
                year == today.year and month > today.month
            ):
                continue

            try:
                month_df = extract_loto_649_month(
                    year=year,
                    month=month
                )

                extraction_log.append({
                    "year": year,
                    "month": month,
                    "status": "success",
                    "draw_count": len(month_df),
                    "error_message": None
                })

                if not month_df.empty:
                    monthly_datasets.append(month_df)

                print(
                    f"{year}-{month:02d}: "
                    f"{len(month_df)} draws extracted"
                )

            except Exception as error:
                extraction_log.append({
                    "year": year,
                    "month": month,
                    "status": "error",
                    "draw_count": None,
                    "error_message": str(error)
                })

                print(
                    f"{year}-{month:02d}: "
                    f"ERROR - {error}"
                )

            # Pause briefly before sending the next request.
            time.sleep(request_delay)

    if monthly_datasets:
        full_history_df = pd.concat(
            monthly_datasets,
            ignore_index=True
        )

        full_history_df = (
            full_history_df
            .drop_duplicates(subset="draw_date")
            .sort_values("draw_date")
            .reset_index(drop=True)
        )
    else:
        full_history_df = pd.DataFrame(
            columns=[
                "draw_date",
                "number_1",
                "number_2",
                "number_3",
                "number_4",
                "number_5",
                "number_6"
            ]
        )

    extraction_log_df = pd.DataFrame(extraction_log)

    return full_history_df, extraction_log_df

## Historical extraction

The complete historical dataset is collected from the first year currently available in the website selector up to the current month.

The extraction log is kept separately from the draw dataset. This allows months with zero results to be distinguished from months where the HTTP request or HTML parsing failed.

In [39]:
full_history_df, full_extraction_log_df = (
    extract_loto_649_history(
        start_year=1998,
        end_year=2026,
        request_delay=0.5
    )
)

1998-01: 4 draws extracted
1998-02: 4 draws extracted
1998-03: 5 draws extracted
1998-04: 4 draws extracted
1998-05: 5 draws extracted
1998-06: 4 draws extracted
1998-07: 4 draws extracted
1998-08: 5 draws extracted
1998-09: 4 draws extracted
1998-10: 4 draws extracted
1998-11: 5 draws extracted
1998-12: 3 draws extracted
1999-01: 5 draws extracted
1999-02: 4 draws extracted
1999-03: 4 draws extracted
1999-04: 4 draws extracted
1999-05: 5 draws extracted
1999-06: 4 draws extracted
1999-07: 4 draws extracted
1999-08: 5 draws extracted
1999-09: 4 draws extracted
1999-10: 5 draws extracted
1999-11: 4 draws extracted
1999-12: 4 draws extracted
2000-01: 5 draws extracted
2000-02: 4 draws extracted
2000-03: 4 draws extracted
2000-04: 5 draws extracted
2000-05: 4 draws extracted
2000-06: 4 draws extracted
2000-07: 5 draws extracted
2000-08: 4 draws extracted
2000-09: 4 draws extracted
2000-10: 5 draws extracted
2000-11: 4 draws extracted
2000-12: 4 draws extracted
2001-01: 5 draws extracted
2

In [40]:
display(full_extraction_log_df)

print("\nExtraction status:")
display(
    full_extraction_log_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="month_count")
)

print("\nMonths with zero extracted draws:")
display(
    full_extraction_log_df[
        full_extraction_log_df["draw_count"].eq(0)
    ]
)

,year,month,status,draw_count,error_message
0,1998,1,success,4,None
1,1998,2,success,4,None
2,1998,3,success,5,None
3,1998,4,success,4,None
4,1998,5,success,5,None
...,...,...,...,...,...
338,2026,3,success,9,None
339,2026,4,success,8,None
340,2026,5,success,9,None
341,2026,6,success,8,None



Extraction status:


,status,month_count
0,success,343



Months with zero extracted draws:


,year,month,status,draw_count,error_message
267,2020,4,success,0,None
268,2020,5,success,0,None


In [41]:
validate_loto_649_data(full_history_df)

print(
    "Full historical dataset validation passed."
)

Full historical dataset validation passed.


## Exporting the collected data

The validated historical dataset is saved locally so that the website does not need to be queried again during every analysis session.

Two formats are generated:

- **CSV**, which is lightweight and convenient for data analysis and machine learning;
- **Excel**, which is useful for manual inspection and includes both the historical draws and the monthly extraction log.

In [42]:
from pathlib import Path


# Create an output directory inside the current Colab session.
output_directory = Path("data")
output_directory.mkdir(parents=True, exist_ok=True)

csv_path = output_directory / "loto_649_historical_draws.csv"
excel_path = output_directory / "loto_649_historical_data.xlsx"
log_csv_path = output_directory / "loto_649_extraction_log.csv"


# Save the main historical dataset as CSV.
full_history_df.to_csv(
    csv_path,
    index=False,
    date_format="%Y-%m-%d"
)

# Save the extraction log separately as CSV.
full_extraction_log_df.to_csv(
    log_csv_path,
    index=False
)

# Save both datasets in one Excel workbook.
with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd"
) as writer:
    full_history_df.to_excel(
        writer,
        sheet_name="Historical Draws",
        index=False
    )

    full_extraction_log_df.to_excel(
        writer,
        sheet_name="Extraction Log",
        index=False
    )

print(f"Historical CSV saved to: {csv_path}")
print(f"Extraction log CSV saved to: {log_csv_path}")
print(f"Excel workbook saved to: {excel_path}")

Historical CSV saved to: data/loto_649_historical_draws.csv
Extraction log CSV saved to: data/loto_649_extraction_log.csv
Excel workbook saved to: data/loto_649_historical_data.xlsx


In [43]:
from google.colab import files

files.download(str(excel_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
files.download(str(csv_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
saved_history_df = pd.read_csv(
    csv_path,
    parse_dates=["draw_date"]
)

print(f"Rows loaded from CSV: {len(saved_history_df)}")
display(saved_history_df.head())

assert len(saved_history_df) == len(full_history_df)
assert list(saved_history_df.columns) == list(full_history_df.columns)

print("Saved dataset verification passed.")

Rows loaded from CSV: 2286


,draw_date,number_1,number_2,number_3,number_4,number_5,number_6
0,1998-01-04,17,23,5,35,34,24
1,1998-01-11,27,32,20,28,29,12
2,1998-01-18,16,33,21,8,23,43
3,1998-01-25,26,46,1,27,30,13
4,1998-02-01,23,10,48,44,19,18


Saved dataset verification passed.
